# Lab 4b: Hardware Benchmarking (Algorithmic Prespective)
## Hardware for Machine Learning Course

This notebook is to benchmark tiling algorithm on a CPU. It shows and highlight how the CPU memory heirarchy interact with different tiles' sizes.


### 1. Install a C++ Compiler (g++)

First, you need to install a C++ compiler. Colab environments are based on Ubuntu, so you can use `apt-get` to install `g++`.

In [1]:
get_ipython().system('apt-get update && apt-get install -y g++')

Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
g++ is already the newest version (4:11.2.0-1ubuntu1).
0 upgraded, 0 newly installed, 0 to rem

### 2. Complete your C++ Code To-Dos

Next, complete your C++ code. We will use the `%%writefile` magic command to create a file directly in a Colab cell.

In [2]:
%%writefile benchmark_naive_vs_tiling.cpp
#include <iostream>
#include <vector>
#include <chrono>
#include <fstream>
#include <string>
#include <algorithm>
#include <cassert>
#include <iomanip>
using namespace std;

class Matrix {
public:
    int n;
    vector<double> data;

    Matrix(int size) : n(size), data(size * size, 0.0) {}

    double& operator()(int i, int j) { return data[i * n + j]; }
    const double& operator()(int i, int j) const { return data[i * n + j]; }
};

// Naive
void naive_multiply(const Matrix& A, const Matrix& B, Matrix& C) {
    int n = A.n;
    for (int i = 0; i < n; i++)
        for (int j = 0; j < n; j++)
            for (int k = 0; k < n; k++)
                C(i, j) += A(i, k) * B(k, j);
}

// Tiled
void tiled_multiply(const Matrix& A, const Matrix& B, Matrix& C, int tile_size) {
    int n = A.n;

    for (int i = 0; i < n; i += tile_size) {
        int imax = min(i + tile_size, n);
        for (int j = 0; j < n; j += tile_size) {
            int jmax = min(j + tile_size, n);
            for (int k = 0; k < n; k += tile_size) {
                int kmax = min(k + tile_size, n);

                for (int it = i; it < imax; it++) {
                    for (int jt = j; jt < jmax; jt++) {
                        double sum = 0;
                        for (int kt = k; kt < kmax; kt++) {
                            sum += A(it, kt) * B(kt, jt);
                        }
                        C(it, jt) += sum;
                    }
                }
            }
        }
    }
}

int main() {
    cout << "MATRIX MULTIPLICATION: TILING SWEEP\n";
    cout << "====================================\n\n";

    vector<int> sizes = {64, 128, 512}; //test larger matrices for larger caches sizes
    vector<int> tile_sizes = {1, 4, 8, 16, 32, 64, 128, 256, 512};

    for (int n : sizes) {
        cout << "\n--- Matrix Size: " << n << "x" << n << " ---\n";
        cout << "Total multiply-add operations: " << (long long)n * n * n << " (SAME for both!)\n";

        Matrix A(n), B(n), C1(n), C2(n);

        // Initialize A and B arbitrary
        for (int i = 0; i < n; i++)
            for (int j = 0; j < n; j++) {
                A(i, j) = i + j;
                B(i, j) = i - j;
            }
        // Initialize C1 and C2 to zeros
        for (int i = 0; i < n * n; i++) C1.data[i] = 0.0;
        for (int i = 0; i < n * n; i++) C2.data[i] = 0.0;

        // Baseline: naive
        auto start = chrono::high_resolution_clock::now();
        naive_multiply(A, B, C1);
        auto end = chrono::high_resolution_clock::now();
        // auto naive_time = chrono::duration_cast<chrono::milliseconds>(end - start).count();
        auto naive_time_us = chrono::duration_cast<chrono::microseconds>(end - start).count();
        double naive_time_ms = naive_time_us / 1000.0;
        // cout << "Naive: " << naive_time << " ms\n";
        if (naive_time_us < 1000) {
            cout << "Naive: " << naive_time_us << " µs\n";
        } else {
            cout << "Naive: " << fixed << setprecision(2) << naive_time_ms << " ms\n";
        }


        // Try different tile sizes
        for (int t : tile_sizes) {
            if (t > n) continue;

            // Reset C
            for (int i = 0; i < n * n; i++) C2.data[i] = 0.0;

            start = chrono::high_resolution_clock::now();
            tiled_multiply(A, B, C2, t);
            end = chrono::high_resolution_clock::now();
            // auto tiled_time = chrono::duration_cast<chrono::milliseconds>(end - start).count();
            auto tiled_time_us = chrono::duration_cast<chrono::microseconds>(end - start).count();
            double tiled_time_ms = tiled_time_us / 1000.0;
            for (int i_test = 0; i_test < n * n; i_test++){
                if (C1.data[i_test] != C2.data[i_test]) {
                    assert(false && "Error: Matrix multiplication results don't match!");
                    exit(1);
                }
            }
            // double speedup = (double)naive_time / tiled_time;
            double speedup = naive_time_us / (double)tiled_time_us;
            string faster = speedup >= 1.0 ? "FASTER" : "SLOWER";

            // cout << "  Tile " << t << ": " << tiled_time << " ms, "
            //      << speedup << "x " << faster << "\n";
            cout << "  Tile " << t << ": ";
            if (tiled_time_us < 1000) {
                cout << tiled_time_us << " µs, ";
            } else {
                cout << fixed << setprecision(2) << tiled_time_ms << " ms, ";
            }
            cout << fixed << setprecision(2) << speedup << "x " << faster << "\n";
        }
    }

    return 0;
}

Overwriting benchmark_naive_vs_tiling.cpp


### 3. Compile the C++ Code

Now, compile your C++ source file using `g++`.

In [3]:
get_ipython().system('g++ -O2 benchmark_naive_vs_tiling.cpp -o naive_vs_tiling')

### 4. Run the Compiled Executable

Finally, you can run the compiled executable.

In [4]:
get_ipython().system('./naive_vs_tiling')

MATRIX MULTIPLICATION: TILING SWEEP


--- Matrix Size: 64x64 ---
Total multiply-add operations: 262144 (SAME for both!)
Naive: 590 µs
  Tile 1: 2.15 ms, 0.27x SLOWER
  Tile 4: 370 µs, 1.59x FASTER
  Tile 8: 287 µs, 2.06x FASTER
  Tile 16: 255 µs, 2.31x FASTER
  Tile 32: 328 µs, 1.80x FASTER
  Tile 64: 312 µs, 1.89x FASTER

--- Matrix Size: 128x128 ---
Total multiply-add operations: 2097152 (SAME for both!)
Naive: 3.50 ms
  Tile 1: 16.48 ms, 0.21x SLOWER
  Tile 4: 2.89 ms, 1.21x FASTER
  Tile 8: 2.56 ms, 1.37x FASTER
  Tile 16: 2.09 ms, 1.68x FASTER
  Tile 32: 2.28 ms, 1.54x FASTER
  Tile 64: 2.51 ms, 1.40x FASTER
  Tile 128: 2.77 ms, 1.27x FASTER

--- Matrix Size: 512x512 ---
Total multiply-add operations: 134217728 (SAME for both!)
Naive: 523.00 ms
  Tile 1: 2534.55 ms, 0.21x SLOWER
  Tile 4: 246.52 ms, 2.12x FASTER
  Tile 8: 164.47 ms, 3.18x FASTER
  Tile 16: 162.35 ms, 3.22x FASTER
  Tile 32: 185.87 ms, 2.81x FASTER
  Tile 64: 225.31 ms, 2.32x FASTER
  Tile 128: 236.83 ms, 2.21x FAS